In [3]:
import numpy as np
import matplotlib.pyplot as plt
import pyroomacoustics as pra
import torchaudio
import torchaudio.transforms
import random
import os
import tqdm

In [5]:
SAMPLE_RATE = 16000

In [6]:
fade_transform = torchaudio.transforms.Fade(fade_in_len=int(0.25 * SAMPLE_RATE), fade_out_len=int(0.25 * SAMPLE_RATE), fade_shape='linear')

## Helper Functions

In [ ]:
def load_audio(file_path):
    sample_rate, data = wavfile.read(file_path)
    return sample_rate, data

def random_room_dimensions():
    return [random.uniform(5, 20) for _ in range(3)]

def random_rt60():
    return random.uniform(0, 1)

def random_distance_and_angle():
    distance = random.uniform(1, 5)
    angle = random.uniform(0, 2 * np.pi)
    return distance, angle

def random_snr():
    return random.uniform(-5, 5)

## Main Function

In [ ]:
def generate_synthetic_example(vctk_path, wham_path):
    # Load random target and secondary speaker from VCTK
    target_speaker_file = random.choice(os.listdir(vctk_path))
    secondary_speaker_file = random.choice(os.listdir(vctk_path))
    
    target_sr, target_audio = load_audio(os.path.join(vctk_path, target_speaker_file))
    secondary_sr, secondary_audio = load_audio(os.path.join(vctk_path, secondary_speaker_file))
    
    # Load random background noise from WHAM!
    background_file = random.choice(os.listdir(wham_path))
    background_sr, background_audio = load_audio(os.path.join(wham_path, background_file))
    
    # Ensure all audio is 3 seconds long
    target_audio = target_audio[:3 * target_sr]
    secondary_audio = secondary_audio[:3 * secondary_sr]
    background_audio = background_audio[:3 * background_sr]
    
    # Create room with random dimensions and RT60
    room_dim = random_room_dimensions()
    rt60 = random_rt60()
    room = pra.ShoeBox(room_dim, fs=target_sr, max_order=12, absorption=pra.inverse_sabine(rt60))
    
    # Place microphones
    mic_distance = 0.175  # 17.5 cm
    mic_center = np.array(room_dim) / 2
    mic_locs = np.array([[mic_center[0] - mic_distance / 2, mic_center[1], mic_center[2]],
                         [mic_center[0] + mic_distance / 2, mic_center[1], mic_center[2]]]).T
    room.add_microphone_array(mic_locs)
    
    # Place target speaker at center
    room.add_source(mic_center, signal=target_audio)
    
    # Place secondary speaker at random distance and angle
    distance, angle = random_distance_and_angle()
    secondary_loc = mic_center + np.array([distance * np.cos(angle), distance * np.sin(angle), 0])
    room.add_source(secondary_loc, signal=secondary_audio)
    
    # Add background noise
    room.add_source(mic_center, signal=background_audio)
    
    # Simulate the room
    room.simulate()
    
    # Mix signals with random SNR
    snr = random_snr()
    mixed_signals = room.mic_array.signals
    mixed_signals = mixed_signals / np.max(np.abs(mixed_signals))  # Normalize
    
    # Adjust background volume based on SNR
    signal_power = np.mean(mixed_signals[0]**2)
    noise_power = np.mean(background_audio**2)
    scaling_factor = np.sqrt(signal_power / (10**(snr / 10) * noise_power))
    mixed_signals += scaling_factor * background_audio[:mixed_signals.shape[1]]
    
    return mixed_signals

In [ ]:
def generate_dataset(vctk_path, wham_path, num_samples=10000):
    dataset = []
    for _ in range(num_samples):
        example = generate_synthetic_example(vctk_path, wham_path)
        dataset.append(example)
    return dataset

## Main Loop

In [ ]:
vctk_path = '/path/to/VCTK'
wham_path = '/path/to/WHAM'
dataset = generate_dataset(vctk_path, wham_path)